# 1. Load the data

In [1]:
passenger_traffic.dtypes

NameError: name 'passenger_traffic' is not defined

In [ ]:
airline_financials = pd.read_csv("data/airline_financials.csv")
aviation_incidents = pd.read_csv("data/aviation_incidents.csv")
fleet_orders = pd.read_csv("data/fleet_orders.csv")
passenger_traffic = pd.read_csv("data/passenger_traffic.csv")
route_performance = pd.read_csv("data/route_performance.csv")

In [ ]:
passenger_traffic.shape

### 2.5 route_performance

In [ ]:
route_performance.head()

In [ ]:
route_performance.dtypes

In [ ]:
route_performance.shape

# 3. Explanatory Data Analysis(EDA)

### 3.1 Statistical Analysis

### 3.1 Statistical Analysis

In [ ]:
def statistical_summary(df: pd.DataFrame, name: str):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(f"Shape: {df.shape}")

    print("\n--- Dtypes ---")
    print(df.dtypes)

    print("\n--- Missing values ---")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    print(pd.DataFrame({"missing": missing, "pct": missing_pct})[missing > 0])

    print("\n--- Duplicates ---")
    print(f"{df.duplicated().sum()} duplicate rows")

    print("\n--- Numeric summary ---")
    print(df.describe().T)

    print("\n--- Categorical summary ---")
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts().head(10))

    if "year" in df.columns:
        print(f"\n--- Year range ---")
        print(f"{df['year'].min()} to {df['year'].max()}, {df['year'].nunique()} unique years")


for name, df in {
    "airline_financials": airline_financials,
    "fleet_orders": fleet_orders,
    "passenger_traffic": passenger_traffic,
    "aviation_incidents": aviation_incidents,
    "route_performance": route_performance,
}.items():
    statistical_summary(df, name)

In [ ]:
import os

def eda_charts(df: pd.DataFrame, name: str, out_dir: str = "charts"):
    os.makedirs(out_dir, exist_ok=True)

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if "year" in numeric_cols:
        numeric_cols.remove("year")

    n_cols = 3
    n_rows = -(-len(numeric_cols) // n_cols)  # ceil division

    # Distributions
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    for i, col in enumerate(numeric_cols):
        sns.histplot(df[col], kde=True, ax=axes[i])
        axes[i].set_title(col)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    plt.suptitle(f"{name} — Distributions")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/{name}_distributions.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Outliers
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    for i, col in enumerate(numeric_cols):
        sns.boxplot(x=df[col], ax=axes[i])
        axes[i].set_title(col)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    plt.suptitle(f"{name} — Outliers")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/{name}_outliers.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Correlations
    if len(numeric_cols) > 1:
        fig = plt.figure(figsize=(8, 6))
        sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", center=0)
        plt.title(f"{name} — Correlations")
        plt.savefig(f"{out_dir}/{name}_correlations.png", dpi=150)
        plt.show()
        plt.close(fig)

    # Yearly trend
    if "year" in df.columns:
        fig = plt.figure(figsize=(10, 4))
        yearly = df.groupby("year")[numeric_cols].mean()
        yearly.plot(ax=plt.gca())
        plt.title(f"{name} — Yearly Trend")
        plt.axvspan(2020, 2021, color="red", alpha=0.1, label="COVID")
        plt.legend()
        plt.savefig(f"{out_dir}/{name}_yearly_trend.png", dpi=150)
        plt.show()
    plt.show()

In [ ]:
eda_charts(airline_financials, "airline_financials")

In [ ]:
eda_charts(airline_financials, "airline_financials")

In [ ]:
eda_charts(fleet_orders, "fleet_orders")

In [ ]:
eda_charts(passenger_traffic, "passenger_traffic")

In [ ]:
eda_charts(route_performance, "route_performance")

# 4. PreProcessing

### 4.1 Checking for missing values

In [ ]:
airline_financials.isna().sum()

In [ ]:
aviation_incidents.isna().sum()

In [ ]:
fleet_orders.isna().sum()

In [ ]:
passenger_traffic.isna().sum()

In [ ]:
route_performance.isna().sum()

### 4.2 Feature engeneering

In [ ]:
route_performance

In [ ]:
airline_financials = airline_financials.rename(columns={
    "avg_fare_usd": "industry_avg_fare_usd",
    "total_route_passengers_m": "industry_total_route_passengers_m",
    "n_routes_reported": "industry_n_routes_reported"})

In [ ]:
route_yearly = route_performance.groupby("year").agg(avg_fare_usd = ("avg_fare_usd","mean"),
                                                    total_route_passengers_m=("annual_passengers_m", "sum"),
                                                    n_routes_reported=("route", "count")).reset_index()

In [ ]:
airline_financials = airline_financials.merge(route_yearly, on="year", how="left")

In [ ]:
obj_cols = airline_financials.select_dtypes(include="object").columns.tolist()
numeric_cols = airline_financials.select_dtypes(include="number").columns.tolist()
numeric_cols.remove("revenue_usd_bn")
numeric_cols.remove("operating_income_usd_bn")

### 4.3 Column Transformer

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), obj_cols),
    ("num", numeric_pipeline, numeric_cols),
])

In [ ]:
airline_financials.columns[airline_financials.columns.duplicated()]

In [ ]:
airline_financials.dtypes

In [ ]:
x = airline_financials.drop(columns=["revenue_usd_bn", "operating_income_usd_bn"])
y = airline_financials["revenue_usd_bn"]

train_mask = airline_financials["year"] <= 2022
x_train, x_test = x[train_mask], x[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

In [ ]:
x = airline_financials.drop(columns=["revenue_usd_bn", "operating_income_usd_bn"])
y = airline_financials["revenue_usd_bn"]

# 5 The Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RepeatedKFold, cross_val_score

In [ ]:
models = {
    "RandomForestRegressor": RandomForestRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "XGBRegressor": XGBRegressor(random_state=42),
    "CatBoostRegressor": CatBoostRegressor(verbose=0, random_state=42),
    "LGBMRegressor": LGBMRegressor(random_state=42)}

In [ ]:
result = []

In [ ]:
best_pipe = None
best_model_name = None
best_r2 = -999

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)])
    pipe.fit(x_train, y_train)
    y_pred = pipe.predict(x_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    result.append({"Name": name, "R2": r2, "MAE": mae, "MSE": mse})

    if r2 > best_r2:
        best_pipe = pipe
        best_model_name = name
        best_r2 = r2

y_pred = best_pipe.predict(x_test)

# 6. Model Result Analysis

In [ ]:
result_table = pd.DataFrame(result).sort_values("R2", ascending=False)

In [ ]:
feat_imp = best_pipe.named_steps["model"].feature_importances_
feat_names = best_pipe.named_steps["preprocessor"].get_feature_names_out()
feat_df = pd.DataFrame({"Features": feat_names, "importance": feat_imp})
feat_df = feat_df.sort_values("importance", ascending=False).head(10)
sns.barplot(x="importance", y="Features", data=feat_df)
plt.title("Feature Importance")
plt.show()

In [ ]:
x_test_transformed = best_pipe.named_steps["preprocessor"].transform(x_test)
explainer = shap.TreeExplainer(best_pipe.named_steps["model"])
shap_values = explainer.shap_values(x_test_transformed)
shap.summary_plot(shap_values, x_test_transformed)

In [ ]:
os.makedirs("trained_model", exist_ok=True)
joblib.dump(best_pipe, "trained_model/final_model.pkl")
joblib.dump(list(x_train.columns), "trained_model/feature_names.pkl")

In [ ]:
print(best_pipe)